In [3]:
# --- Imports ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

# --- Load processed dataset ---
df = pd.read_csv("../data/processed_hr_dataset.csv")

# --- Define target and features ---
# Target: Attrition encoded as 1 (Yes) / 0 (No)
# If Attrition column is not in processed dataset, we can reload original df for target
df_original = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
y = df_original["Attrition"].map({"Yes":1, "No":0})
X = df.drop(columns=["MonthlyIncome"])  # Drop target used for regression

# --- Split into train and test ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

joblib.dump(X_train.columns, "../models/classification_feature_names.joblib")

# --- Train Random Forest Classifier ---
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# --- Predict on train and test ---
y_train_pred = rf.predict(X_train)
y_test_pred = rf.predict(X_test)

# --- Evaluate model ---
def classification_metrics(y_true, y_pred, dataset=""):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"--- {dataset} set metrics ---")
    print(f"Accuracy: {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall: {rec:.3f}")
    print(f"F1-score: {f1:.3f}\n")
    
classification_metrics(y_train, y_train_pred, "Train")
classification_metrics(y_test, y_test_pred, "Test")

# --- Confusion matrix ---
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No","Yes"], yticklabels=["No","Yes"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# --- Classification report (optional) ---
print(classification_report(y_test, y_test_pred, target_names=["No Attrition", "Attrition"]))

# --- Save trained model ---
joblib.dump(rf, "../models/rf_attrition_model.joblib")
print("Random Forest model saved as 'rf_attrition_model.joblib'")


ModuleNotFoundError: No module named 'sklearn'